In [17]:
import pandas as pd
from matplotlib import pyplot
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

In [18]:
load_dotenv()
database_url = os.getenv('DATABASE_URL')

engine = create_engine(database_url)

ranking_stdev_df = pd.read_sql('''
                SELECT
                    t.full_name AS team_name,
                    g.year,
                    STDDEV(g.ranking) AS ranking_stdev
                FROM (
                        SELECT
                            year,
                            id_home_team AS team_id,
                            home_team_ranking AS ranking
                        FROM
                            games
                        WHERE
                        home_team_ranking != -1
                    UNION ALL
                        SELECT
                            year,
                            id_away_team AS team_id,
                            away_team_ranking AS ranking
                        FROM
                            games
                        WHERE
                            away_team_ranking != -1
                ) AS g
                JOIN
                    teams AS t ON g.team_id = t.team_id
                GROUP BY
                    t.full_name,
                    g.year
                HAVING
                    COUNT(g.ranking) >= 2
                ORDER BY
                t.full_name,
                g.year;'''
                , engine)

print(ranking_stdev_df)

              team_name  year  ranking_stdev
0     Air Force Falcons  1958       2.768875
1     Air Force Falcons  1959       0.577350
2     Air Force Falcons  1969       0.577350
3     Air Force Falcons  1970       3.973396
4     Air Force Falcons  1971       1.414214
...                 ...   ...            ...
2705      Yale Bulldogs  1937       4.082483
2706      Yale Bulldogs  1944       1.414214
2707      Yale Bulldogs  1946       3.000000
2708      Yale Bulldogs  1959       4.242641
2709      Yale Bulldogs  1960       2.081666

[2710 rows x 3 columns]


In [19]:
top_five_teams_count_df = pd.read_sql('''
    SELECT * FROM games LIMIT 10
                                      ''', engine)

print(top_five_teams_count_df)

   game_id  year  week  postseason  id_home_team  id_away_team  points_home  \
0    21258  1936     6           0          2633           150           15   
1    21229  1936     6           0           108           159            7   
2    21245  1936     6           0           127       1000039            7   
3    21238  1936     6           0           344          2628            0   
4    21149  1936     5           0           221          2184            0   
5    21140  1936     5           0          2102           218            7   
6    21089  1936     4           0           194           221            0   
7    21096  1936     4           0           135           158            7   
8    21074  1936     4           0            26           264            0   
9    21084  1936     4           0           356            30            6   

   points_away  completed  conference_game  home_team_ranking  \
0           13          1                0                 17   
